In [2]:
import torch
import tifffile as tif
import random
from torchvision import transforms
import os

random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

path = "../../data/"
limit = 100
regions = [f for f in os.listdir(path) if (os.path.isdir(os.path.join(path, f)) and f != "study areas shp")]

regions_dict = dict()

for region in regions:
    dataset_dir = path + region
    image_dir = os.path.join(dataset_dir, "img")
    img_list = os.listdir(image_dir)

    all_images = sorted(os.path.join(image_dir, f) for f in img_list)

    if len(all_images) > limit:
        all_images = random.sample(all_images, limit)

    regions_dict[region] = all_images

In [3]:
import torch
from pytorch_wavelets import DWTForward, DWTInverse

wave, levels, mode, device = 'haar', 1, 'reflect', 'cuda'

dwt = DWTForward(J=levels, wave=wave, mode=mode).to(device)
idwt = DWTInverse(wave=wave, mode=mode).to(device)

@torch.no_grad()
def wavelet_adapt(src, tgt, method, alpha):
    src = src.unsqueeze(0)
    tgt = tgt.unsqueeze(0)

    s_LL, s_H = dwt(src)
    t_LL, t_H = dwt(tgt)

    if method == 'replace':
        new_LL = (1 - alpha) * s_LL + alpha * t_LL
    elif method == 'match':
        mu_s, sd_s = s_LL.mean((2,3), keepdim=True), s_LL.std((2,3), keepdim=True).clamp_min(1e-6)
        mu_t, sd_t = t_LL.mean((2,3), keepdim=True), t_LL.std((2,3), keepdim=True).clamp_min(1e-6)
        t_LL_ms = (t_LL - mu_t) / sd_t * sd_s + mu_s
        new_LL = (1 - alpha) * s_LL + alpha * t_LL_ms
    
    # if method == 'coral':
    #     new_LL = coral_align(s_LL, t_LL)

    stylized = idwt((new_LL, s_H)).clamp(0,1)
    stylized = stylized.squeeze(0)
    return stylized

In [4]:
to_tensor = transforms.ToTensor()

def load_img(path):
    img = tif.imread(path).astype("float32")
    return to_tensor(img).float().to(device)


In [ ]:
@torch.no_grad()
def extract_LL_HF(img):
    x = img.unsqueeze(0)
    LL, HF = dwt(x)
    return LL.squeeze(0), HF


In [6]:
def get_region_images(region_name, n=5):
    img_paths = regions_dict[region_name]
    chosen = random.sample(img_paths, n)
    return [load_img(p) for p in chosen], chosen


In [ ]:
def prepare_wavelet_diagram(
    src_region, tgt_region,
    n=5,
    alpha=1.0,
    method='replace'
):
    src_imgs, src_paths = get_region_images(src_region, n=n)
    tgt_imgs, tgt_paths = get_region_images(tgt_region, n=n)
    
    outputs = []
    
    for i in range(n):
        src = src_imgs[i]
        tgt = tgt_imgs[i]

        s_LL, s_H = extract_LL_HF(src)
        t_LL, t_H = extract_LL_HF(tgt)

        adapted = wavelet_adapt(src, tgt, method=method, alpha=alpha)

        outputs.append({
            "src_img": src.cpu(),
            "tgt_img": tgt.cpu(),
            "src_LL": s_LL.cpu(),
            "src_HF": s_H[0].cpu(),
            "tgt_LL": t_LL.cpu(),
            "tgt_HF": t_H[0].cpu(),
            "adapted": adapted.cpu(),
            "src_path": src_paths[i],
            "tgt_path": tgt_paths[i]
        })
    
    return outputs


In [ ]:
import torch

def normalize_for_save(tensor):
    x = tensor.clone().detach().cpu()

    if x.min() < 0:
        x = torch.abs(x)

    x = x - x.min()
    if x.max() > 0:
        x = x / x.max()

    return x


In [ ]:
src_region = "Jiuzhai valley (UAV-0.2m)"
tgt_region = "Hokkaido Iburi-Tobu"

import os
import torchvision.utils as vutils

def save_wavelet_sample(sample, outdir, index):
    os.makedirs(outdir, exist_ok=True)

    prefix = f"{outdir}/sample_{index}_"

    items = {
        "src_img":   "source_rgb.png",
        "tgt_img":   "target_rgb.png",
        "adapted":   "adapted_rgb.png",
        "src_LL":    "source_LL.png",
        "src_HF":    "source_HF.png",
        "tgt_LL":    "target_LL.png",
        "tgt_HF":    "target_HF.png",
    }

    for key, fname in items.items():
        x = sample[key].clone().detach().cpu()

        while x.ndim > 3:
            x = x.squeeze(0)

        if x.ndim == 4:
            x = x.abs().mean(dim=1)

        x = x - x.min()
        if x.max() > 0:
            x = x / x.max()

        if x.ndim == 2:
            x = x.unsqueeze(0) 
        elif x.shape[0] not in [1, 3]:
            x = x.mean(dim=0, keepdim=True)

        vutils.save_image(x, prefix + fname)

    print(f"[✓] Saved sample {index} to {outdir}")



In [15]:
samples = prepare_wavelet_diagram(src_region, tgt_region, n=5)


In [ ]:
for i, sample in enumerate(samples):
    save_wavelet_sample(sample, outdir="wavelet_outputs", index=i)


KeyboardInterrupt: 

In [ ]:
import torch
import tifffile as tif
import random
from torchvision import transforms
import torchvision.utils as vutils
import os

random.seed(42)
device = torch.device("cpu")

path = "../../data/"
limit = 100
regions = [
    f for f in os.listdir(path)
    if (os.path.isdir(os.path.join(path, f)) and f != "study areas shp")
]

regions_dict = {}
for region in regions:
    dataset_dir = os.path.join(path, region)
    image_dir = os.path.join(dataset_dir, "img")
    img_list = os.listdir(image_dir)

    all_images = sorted(os.path.join(image_dir, f) for f in img_list)
    if len(all_images) > limit:
        all_images = random.sample(all_images, limit)

    regions_dict[region] = all_images

print("Regions:", list(regions_dict.keys()))

from pytorch_wavelets import DWTForward, DWTInverse

wave = 'haar'
levels = 1
mode = 'reflect'

dwt = DWTForward(J=levels, wave=wave, mode=mode).to(device)
idwt = DWTInverse(wave=wave, mode=mode).to(device)

to_tensor = transforms.ToTensor()

def load_img(path):
    arr = tif.imread(path)
    arr = arr.astype("float32") / 255.0
    img = to_tensor(arr).float()     
    return img.to(device)

src_region = "palu"
tgt_region = "Hokkaido Iburi-Tobu"
random.seed()

src_path = "../../data/palu\img\Palu0870.tif"
tgt_path = "../../data/Hokkaido Iburi-Tobu\img\Hokkaido1546.tif"

print("Source path:", src_path)
print("Target path:", tgt_path)

src_img = load_img(src_path) 
tgt_img = load_img(tgt_path)   

def normalize_for_save(x):
    x = x.clone().detach().cpu()
    x = x - x.min()
    if x.max() > 0:
        x = x / x.max()
    return x

def save_img(tensor, path):
    x = tensor
    if x.ndim == 2:
        x = x.unsqueeze(0)
    if x.shape[0] not in [1,3]:
        x = x.mean(dim=0, keepdim=True)
    x = normalize_for_save(x)
    vutils.save_image(x, path)

outdir = "05"
os.makedirs(outdir, exist_ok=True)

@torch.no_grad()
def decompose_reconstruct(img):
    """
    img: [3,H,W] in 0–1
    returns:
      img_LL: LL-only reconstruction [3,H,W]
      img_HF: HF-only reconstruction [3,H,W] (original - LL)
    """
    x = img.unsqueeze(0)     
    LL, Yh = dwt(x)          


    zero_Yh = [torch.zeros_like(h) for h in Yh]
    x_LL = idwt((LL, zero_Yh))   
    x_LL = x_LL.squeeze(0)

    x_HF = (img - x_LL).clamp(-1, 1)

    return x_LL, x_HF, LL, Yh

src_LL_img, src_HF_img, s_LL_coeff, s_Yh = decompose_reconstruct(src_img)
tgt_LL_img, tgt_HF_img, t_LL_coeff, t_Yh = decompose_reconstruct(tgt_img)

@torch.no_grad()
def ll_replace_adapt(src_img, tgt_img, alpha=.10):
    xs = src_img.unsqueeze(0)
    xt = tgt_img.unsqueeze(0)

    s_LL, s_Yh = dwt(xs)
    t_LL, t_Yh = dwt(xt)

    new_LL = (1 - alpha) * s_LL + alpha * t_LL

    x_adapt = idwt((new_LL, s_Yh)).squeeze(0)   # [3,H,W]
    x_adapt = x_adapt.clamp(0,1)
    return x_adapt

adapted_img = ll_replace_adapt(src_img, tgt_img, alpha=0.5)

save_img(src_img,        os.path.join(outdir, "01_source_rgb.png"))
save_img(src_LL_img,     os.path.join(outdir, "02_source_LL_only.png"))
save_img(src_HF_img,     os.path.join(outdir, "03_source_HF_only.png"))

save_img(tgt_img,        os.path.join(outdir, "04_target_rgb.png"))
save_img(tgt_LL_img,     os.path.join(outdir, "05_target_LL_only.png"))
save_img(tgt_HF_img,     os.path.join(outdir, "06_target_HF_only.png"))

save_img(adapted_img,    os.path.join(outdir, "07_adapted_ll_transfer.png"))

print("Saved images to:", outdir)


Regions: ['Hokkaido Iburi-Tobu', 'Jiuzhai valley (UAV-0.2m)', 'Jiuzhai valley (UAV-0.5m)', 'Lombok', 'Longxi River (SAT)', 'Longxi River (UAV)', 'Mengdong Township', 'Moxi town (UAV-0.2m)', 'Moxi town (UAV-1m)', 'Moxitaidi (SAT)', 'Moxitaidi (UAV-0.6m)', 'Moxitaidi (UAV-1m)', 'palu', 'Tiburon Peninsula (planet)', 'Tiburon Peninsula (Sentinel)', 'Wenchuan']
Source path: ../../data/palu\img\Palu0870.tif
Target path: ../../data/Hokkaido Iburi-Tobu\img\Hokkaido1546.tif
Saved images to: 05


In [ ]:
import cv2
import numpy as np

img = cv2.imread("05/01_source_rgb.png")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)


small = cv2.resize(img, (64, 64), interpolation=cv2.INTER_LINEAR)

symbolic_LL = cv2.resize(small, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)

cv2.imwrite("symbolic_LL.png", cv2.cvtColor(symbolic_LL, cv2.COLOR_RGB2BGR))

sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=7)
sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=7)

mag = np.sqrt(sobel_x**2 + sobel_y**2)
mag = (mag / mag.max()) * 255
mag = mag.astype(np.uint8)

_, edges = cv2.threshold(mag, 80, 255, cv2.THRESH_BINARY)

kernel = np.ones((5,5), np.uint8)
edges_clean = cv2.morphologyEx(edges, cv2.MORPH_OPEN, kernel)

symbolic_HF = cv2.cvtColor(edges_clean, cv2.COLOR_GRAY2RGB)

cv2.imwrite("symbolic_HF.png", cv2.cvtColor(symbolic_HF, cv2.COLOR_RGB2BGR))

print("Saved symbolic_LL.png and symbolic_HF.png")


Saved symbolic_LL.png and symbolic_HF.png


In [ ]:
import cv2
import numpy as np

img = cv2.imread("05/01_source_rgb.png")   # ← your path
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)


symbolic_LL = cv2.GaussianBlur(img, (15, 15), sigmaX=5, sigmaY=5)

cv2.imwrite("symbolic_LL.png", cv2.cvtColor(symbolic_LL, cv2.COLOR_RGB2BGR))

sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=5)
sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=5)

mag = np.sqrt(sobel_x**2 + sobel_y**2)
mag = (mag / mag.max()) * 255
mag = mag.astype(np.uint8)

_, edges = cv2.threshold(mag, 40, 255, cv2.THRESH_BINARY)

kernel = np.ones((3,3), np.uint8)
edges_clean = cv2.morphologyEx(edges, cv2.MORPH_OPEN, kernel)

symbolic_HF = cv2.cvtColor(edges_clean, cv2.COLOR_GRAY2RGB)

cv2.imwrite("symbolic_HF.png", cv2.cvtColor(symbolic_HF, cv2.COLOR_RGB2BGR))

print("Saved: symbolic_LL.png and symbolic_HF.png")


Saved: symbolic_LL.png and symbolic_HF.png


In [ ]:
import cv2
import numpy as np

img = cv2.imread("05/01_source_rgb.png")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

h, w = img.shape[:2]

LL_small = cv2.resize(img, (96, 96), interpolation=cv2.INTER_LINEAR)

symbolic_LL = cv2.resize(LL_small, (w, h), interpolation=cv2.INTER_NEAREST)

cv2.imwrite("symbolic_LL.png", cv2.cvtColor(symbolic_LL, cv2.COLOR_RGB2BGR))

sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=5)
sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=5)

mag = np.sqrt(sobel_x**2 + sobel_y**2)
mag = (mag / mag.max()) * 255
mag = mag.astype(np.uint8)

_, edges = cv2.threshold(mag, 40, 255, cv2.THRESH_BINARY)

kernel = np.ones((3,3), np.uint8)
edges_clean = cv2.morphologyEx(edges, cv2.MORPH_OPEN, kernel)

symbolic_HF = cv2.cvtColor(edges_clean, cv2.COLOR_GRAY2RGB)

cv2.imwrite("symbolic_HF.png", cv2.cvtColor(symbolic_HF, cv2.COLOR_RGB2BGR))

print("Saved final symbolic_LL.png and symbolic_HF.png")


Saved final symbolic_LL.png and symbolic_HF.png


: 